In [1]:
# Importing the libraries
import numpy as np 
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Tapas
[nltk_data]     Mahapatra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [4]:
# Load data
true_df = pd.read_csv('True.csv')
fake_df = pd.read_csv('Fake.csv')

# Label: 1 = real, 0 = fake
true_df['label'] = 1
fake_df['label'] = 0

# Combine + shuffle
news_dataset = pd.concat([true_df, fake_df], ignore_index=True)
news_dataset = news_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

In [5]:
news_dataset.columns

Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')

In [6]:
news_dataset.shape

(44898, 5)

In [7]:
news_dataset.head()

,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",0


In [8]:
# Counting the number of the missing values in the datset
news_dataset.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [9]:
(news_dataset == '').sum()        # empty strings instead of NaN
news_dataset['text'].str.strip().eq('').sum()   # whitespace-only text
news_dataset.duplicated().sum()   # duplicate rows, common in this dataset

np.int64(209)

In [10]:
# Drop exact duplicate rows
news_dataset = news_dataset.drop_duplicates().reset_index(drop=True)

# Confirm
news_dataset.duplicated().sum()   # should be 0
news_dataset.isnull().sum()       # should be all 0 (you already confirmed this)

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [11]:
news_dataset['content'] = news_dataset['title'] + " " + news_dataset['text']

In [12]:
print(news_dataset['content'])

0         BREAKING: GOP Chairman Grassley Has Had Enoug...
1         Failed GOP Candidates Remembered In Hilarious...
2         Mike Pence’s New DC Neighbors Are HILARIOUSLY...
3        California AG pledges to defend birth control ...
4        AZ RANCHERS Living On US-Mexico Border Destroy...
                               ...                        
44684    New York protesters camp out at Goldman Sachs ...
44685    Boiler Room #62 – Fatal Illusions Tune in to t...
44686    ATHEISTS SUE GOVERNOR OF TEXAS Over Display on...
44687    Republican tax plan would deal financial hit t...
44688    U.N. refugee commissioner says Australia must ...
Name: content, Length: 44689, dtype: object


In [13]:
# Separating the data and table
X = news_dataset.drop(columns='label', axis=1)
Y = news_dataset['label']

In [14]:
print(X)
print(Y)

                                                   title  \
0       BREAKING: GOP Chairman Grassley Has Had Enoug...   
1       Failed GOP Candidates Remembered In Hilarious...   
2       Mike Pence’s New DC Neighbors Are HILARIOUSLY...   
3      California AG pledges to defend birth control ...   
4      AZ RANCHERS Living On US-Mexico Border Destroy...   
...                                                  ...   
44684  New York protesters camp out at Goldman Sachs ...   
44685                  Boiler Room #62 – Fatal Illusions   
44686  ATHEISTS SUE GOVERNOR OF TEXAS Over Display on...   
44687  Republican tax plan would deal financial hit t...   
44688  U.N. refugee commissioner says Australia must ...   

                                                    text          subject  \
0      Donald Trump s White House is in chaos, and th...             News   
1      Now that Donald Trump is the presumptive GOP n...             News   
2      Mike Pence is a huge homophobe. He suppor

In [15]:
# Stemming
port_stem = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [16]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stop_words]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [19]:
!pip install swifter

  Using cached swifter-1.4.0.tar.gz (1.2 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16599 sha256=e096365ee23caedeb1e11b6983165ed8e5183f978763daba64115115656ea25b
  Stored in directory: c:\users\tapas mahapatra\appdata\local\pip\cache\wheels\e8\a1\09\564d4c4662764b4113f97a9b2b9a5cf15a5dbf86428c8d3658
Successfully built swifter


In [20]:
import swifter

In [21]:
news_dataset['content'] = news_dataset['content'].swifter.apply(stemming)

Pandas Apply:   0%|          | 0/44689 [00:00<?, ?it/s]

In [31]:
# check one row
print(news_dataset['content'][0])

break gop chairman grassley enough demand trump jr testimoni donald trump white hous chao tri cover russia problem mount hour refus acknowledg problem surround fake news hoax howev fact bear thing differ seem crack congression public leadership chuck grassley r iowa head senat judiciari committe fed demand donald trump jr former trump campaign manag paul manafort testifi committe regard infam shadi meet donald trump shadi russian lawyer promis dirt democrat presidenti nomine hillari clinton fact inform due well demand send signal team trump notabl fire special counsel robert mueller circumst despit fact seem seem trump white hous lay groundwork speak speak tweet regard grassley warn also anyon think senat grassley rest senat seriou need look warn alreadi given trump jr manafort either follow order serv subpoena forc compli refus held contempt congress carri seriou jail time even cruel craven creatur within gop sick donald trump corrupt scandal ridden white hous angri stage hostil takeo

In [32]:
# check all
print(news_dataset['content'])

0        break gop chairman grassley enough demand trum...
1        fail gop candid rememb hilari mock eulog video...
2        mike penc new dc neighbor hilari troll homopho...
3        california ag pledg defend birth control insur...
4        az rancher live us mexico border destroy nanci...
                               ...                        
44684    new york protest camp goldman sach oppos trump...
44685    boiler room fatal illus tune altern current ra...
44686    atheist sue governor texa display capitol grou...
44687    republican tax plan would deal financi hit u u...
44688    u n refuge commission say australia must stop ...
Name: content, Length: 44689, dtype: object


In [34]:
# Shuffle dataset to mix real and fake one
news_dataset = news_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

In [35]:
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [37]:
print(X)

['factbox trump fill top job administr reuter presid elect donald trump nomin high frequenc trade expert vincent viola secretari armi senior transit offici said monday list republican trump select top job administr senat confirm requir post except nation secur advis white hous chief staff white hous director nation econom council white hous strategist tillerson spent entir career exxon mobil corp rose serv chairman ceo civil engin train texan join world largest energi compani led sever oper unit state well yemen thailand russia exxon chief execut maintain close tie moscow oppos u sanction russia incurs crimea mnuchin success privat equiti investor hedg fund manag hollywood financi spent year goldman sach leav assembl investor group buy fail california mortgag lender rebrand onewest bank built southern california largest bank hous advocaci group critic bank foreclosur practic accus quick foreclos struggl homeown matti retir marin gener known tough talk distrust iran battlefield experi i

In [38]:
print(Y)

[1 1 0 ... 0 1 1]


In [39]:
Y.shape

(44689,)

In [41]:
# Converting the textual  data into numeric data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [42]:
print(X.shape)

(44689, 89868)


In [43]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6866618 stored elements and shape (44689, 89868)>
  Coords	Values
  (0, 320)	0.03405638146141964
  (0, 334)	0.026798065260581386
  (0, 454)	0.03367304459329064
  (0, 494)	0.013500208497575448
  (0, 502)	0.018867142041616745
  (0, 521)	0.016776501557004216
  (0, 624)	0.0199332374488807
  (0, 627)	0.022182375862668472
  (0, 660)	0.014876167077171553
  (0, 710)	0.02441724251125702
  (0, 821)	0.07373063418078424
  (0, 901)	0.022455481207639853
  (0, 922)	0.0199332374488807
  (0, 929)	0.046948114256253635
  (0, 930)	0.02981357003477365
  (0, 1065)	0.05422999858599885
  (0, 1105)	0.023517698470191712
  (0, 1195)	0.0672196246819667
  (0, 1197)	0.04427545075355339
  (0, 1285)	0.02039166921491436
  (0, 1487)	0.04392770774580558
  (0, 1702)	0.020466198085444837
  (0, 1709)	0.02869403409187509
  (0, 2144)	0.018965562817082098
  (0, 2242)	0.01895826506963045
  :	:
  (44688, 78624)	0.018234071636097006
  (44688, 78645)	0.0185807533707429

In [45]:
# Spliting the dataset to training and test datat
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

In [49]:
# Training the model - Logistic Regression
model = LogisticRegression()
model.fit(X_train, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [52]:
# Evaluation - accuracy score
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [57]:
print('Accuracy score of the traininng data: ', training_data_accuracy)

Accuracy score of the traininng data:  0.9919722525244049


In [55]:
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [58]:
print('Accuracy score of the test data: ', test_data_accuracy)

Accuracy score of the test data:  0.9860147684045648


In [71]:
# Making a predictive system
X_new = X_test[45]


prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
    print('The news is fake')
else:
    print('The news is real')

[0]
The news is fake


In [72]:
print(Y_test[45])

0


In [81]:
import joblib
joblib.dump(model, 'fake_news_model.joblib')
joblib.dump(vectorizer, 'vectorizer.joblib')

['vectorizer.joblib']

In [80]:
# Disclaimer -->
# Model is trained on ISOT dataset (Reuters + American fake news sources)
# Accuracy may vary for non-Reuters style articles including Indian news sources.